<a href="https://colab.research.google.com/github/Annpeng1005/MLH-Crypto-Tracker/blob/main/Sprint_02_Alerts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 2: Event Detection & Alert Generation

Welcome to Sprint 2! You now have a working pipeline that ingests live market data and joins two databases together.

However, as a Product Manager, you know that stakeholders don't want to read raw spreadsheets. They want actionable insights. Today, we are building the logic engine that analyzes the data and triggers alerts when the market moves.

---

## The Problem (Business Logic)
The trading team wants an automated daily brief. Specifically, they need to know immediately if any of our target coins have dropped in price by **more than 5%** in the last 24 hours.

To get this data, we need to update our CoinGecko API request to ask for the 24-hour change.

**The New Live Prices Endpoint:**
`https://api.coingecko.com/api/v3/simple/price`

   * Use the `params` argument in the get method to include the price history of the targeted coins over the last 24 hours (refer to the previous sprint's notebook) by setting the parameter to `include_24hr_change=true`.

---

## Your Acceptance Criteria (The Ticket)
Please write a Python script below that accomplishes the following:

1. **Fetch & Join:** Re-use your logic from Sprint 1 to fetch the Roster and the new Prices endpoint, and join them together.
2. **The Math Logic:** As you join the data, look at the new `usd_24h_change` value. Write an `if` statement to check if the drop is greater than 5% (Hint: a drop is a negative number, like `-5.1`).
3. **Format the Alert:** If a coin triggers the alert, format a clean string message. Example: *"🚨 ALERT: bitcoin is down -6.2% today. Current price: $65000"*
4. **Export to Text:** Instead of a CSV, export all of your triggered alerts into a text file named `market_alerts.txt`. (If no coins are crashing, the file should just say "Market is stable today.")

---

## Version Control
When you are finished, use **File > Save a copy in GitHub** to push this new notebook to your `MLH-Crypto-Tracker` repository. Make sure your commit message says "Completed Sprint 2 Alerts"!

In [1]:
import requests
import csv

# Step 1: define the URLs
url_roster = "https://api.coingecko.com/api/v3/coins/list"
url_price = "https://api.coingecko.com/api/v3/simple/price"

#Step 2: define the target coin IDs
target_coins = ["bitcoin", "ethereum", "solana", "ripple", "dogecoin"]

#Step 3: make GET request to coin roster
roster_response = requests.get(url_roster)
roster_data = roster_response.json()
# print(roster_data)

In [2]:

# Build your alert pipeline below!
url_live = "https://api.coingecko.com/api/v3/simple/price"
live_response = requests.get(url_live)
live_data = live_response.json()
# print(live_data)

target_coins = ["bitcoin", "ethereum", "solana", "ripple", "dogecoin"]

price_params= {'ids': 'bitcoin,ethereum,solana,ripple,dogecoin',
               'vs_currencies': 'usd',
                'include_24hr_change':True
               }



price_response = requests.get(url_live, params=price_params)

price_data = price_response.json()

print(price_data)

{'bitcoin': {'usd': 78451, 'usd_24h_change': 0.14079179611756304}, 'dogecoin': {'usd': 0.097511, 'usd_24h_change': 1.3586473691969678}, 'ethereum': {'usd': 2335.8, 'usd_24h_change': -1.4337233241326455}, 'ripple': {'usd': 1.44, 'usd_24h_change': 0.8424602933650877}, 'solana': {'usd': 86.25, 'usd_24h_change': -0.7234472010409314}}


In [3]:
filter_coins=[]
for el in roster_data:
  # print(el)
  if el['id'] in target_coins:
    filter_coins.append(el)
print(filter_coins)


[{'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin'}, {'id': 'dogecoin', 'symbol': 'doge', 'name': 'Dogecoin'}, {'id': 'ethereum', 'symbol': 'eth', 'name': 'Ethereum'}, {'id': 'ripple', 'symbol': 'xrp', 'name': 'XRP'}, {'id': 'solana', 'symbol': 'sol', 'name': 'Solana'}]


In [13]:
from os import times_result
#inner loop to cross reference the filter roster with the live pricing data
#match through their id information
filter_price=[]
alerts = []
threshold = 0
for coin in filter_coins:
  # print(coin)
  for price_id in price_data:
    # print(price_id)
    if coin['id'] == price_id:
      change = price_data[price_id]['usd_24h_change']
      price = price_data[price_id]['usd']
      filter_price.append([coin['name'], coin['symbol'], price_data[price_id]['usd']]) #using list

      if change <= threshold:
        # print(price_data)
        alert = f"🚨 ALERT: {coin['name']} is down {change}% today. Current price: {price}"

        alerts.append(alert)
print(alerts)



      # print(price_data[price_id]['usd_24h_change'])
      # print(type(coin['name']))




['🚨 ALERT: Ethereum is down -1.4337233241326455% today. Current price: 2335.8', '🚨 ALERT: Solana is down -0.7234472010409314% today. Current price: 86.25']


In [29]:

with open('market_alerts.txt', "w") as file:
  if len(alerts) == 0:
    file.write('Market is stable today.')
  else:
    for alert in alerts:
      file.write(alert)
      if alert not in alerts[-1]:
        file.write("\n")


print("market_alerts.txt is imported successfully.")

market_alerts.txt is imported successfully.
